# 01 · EDA — the data behind the argument

> **Synthetic data.** IBM HR Analytics benchmark; every number is illustrative of method, not a real workforce.

A *thin* notebook — all logic lives in `src/`. This one loads the cleaned frame and shows the shape of the problem: the target balance and the treatment (`OverTime`) prevalence the whole causal argument rests on.

In [1]:
import os, sys
from pathlib import Path

# Make the repo root the working dir and importable, whether this notebook is
# launched from notebooks/ or from the repo root.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image
pd.set_option("display.width", 120)
print("repo root:", ROOT)

repo root: /Users/jimmy/Documents/hr-attrition-analytics


In [2]:
from src.config import TARGET
from src.causal.common import TREATMENT
from src.data.load import load_clean, train_test

df = load_clean()
print('clean frame:', df.shape)
print(f'\n{TARGET} balance (1 = leaves):')
print(df[TARGET].value_counts(normalize=True).round(3).to_string())

clean frame: (1470, 31)

Attrition balance (1 = leaves):
Attrition
0    0.839
1    0.161


## Stratified split + the treatment

The split is fixed-seed and stratified, so the ~16% base rate is preserved in both folds (no leakage). `OverTime` is the binary intervention studied in Phases 3–5b.

In [3]:
train_df, test_df = train_test(df)
print(f'train/test rows: {len(train_df)} / {len(test_df)}')
print(f'train {TARGET}: {train_df[TARGET].mean():.3f}   test {TARGET}: {test_df[TARGET].mean():.3f}')

on_ot = (df[TREATMENT].astype(str).str.strip() == 'Yes').mean()
print(f'\n{TREATMENT} prevalence (the lever): {on_ot:.3f}')
print('\nraw attrition by overtime (Phase 3 adjusts this causally):')
print(df.groupby(TREATMENT)[TARGET].mean().round(3).to_string())

train/test rows: 1102 / 368
train Attrition: 0.162   test Attrition: 0.160

OverTime prevalence (the lever): 0.283

raw attrition by overtime (Phase 3 adjusts this causally):
OverTime
No     0.104
Yes    0.305


## Feature mix

In [4]:
print(df.dtypes.value_counts().to_string())

int64     24
object     7


**Takeaway.** ~16% leave; ~25% are on overtime, and raw attrition is much higher among them. Whether that gap is *causal* — and whether the highest-risk employees are the ones the lever can move — is Phases 3–4. Next: `02_prediction.ipynb`.